# Save global mortality in single file

Due to memory constraints, global mortality is saved on an annual basis. This script combines yearly files into one.

In [ ]:
import os
import glob
import xarray as xr
from utils.utils import get_scenario_config

In [ ]:
# Number of samples
n_samples = 300

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]
# {years.stop - 1} from OSDMA8 calculation
dates = f"{years.start}-{years.stop - 1}"

MORT_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/global/{n_samples}_samples/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/"

for ens_num in ensemble_members:
    print(f"Processing ensemble number {ens_num:02d}")

    files = f"Global_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_*.nc"
    file_path = os.path.join(MORT_DIR, files)

    ds = xr.open_mfdataset(
        sorted(glob.glob(file_path)),
        combine="nested",
        concat_dim="year")
    ds = ds.assign_coords(year=years)  # Last year removed from OSDMA8

    description = ("Global mortality (COPD) due to ozone "
                   " - scripts by A.F. Wells (2025)")
    ds.attrs["description"] = description
    ds.attrs["model"] = model
    ds.attrs["scenario"] = scenario
    ds.attrs["ensemble_number"] = ens_num

    out_file = f"Global_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving {out_file}")
    ds.to_netcdf(out_path)

print("All processing complete.")